In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/raw/project-db.db')

In [4]:
query = """
SELECT name FROM sqlite_master WHERE type='table';
"""
pd.read_sql_query(query, conn)    # List all tables in the database

,name
0,departments
1,regions
2,employees
3,customers
4,sales


**Rank, Dense_Rank, and Row_Number Window Functions**

In [ ]:
# Retrieve a list of employee_id, first_name, hire_date, and department of all employees ordered by the hire date
query = """
SELECT employee_id,
       first_name,
       hire_date,
       department,
       -- Use DENSE_RANK() to return consecutive integers regardless you ties or not. As shown, there is a tie in the 1st 2 hire dates, and the 
       -- 3rd rank of the hire date is incremented to 2, not to 3 like the RANK() Function. So for RANK() Function, you would get 1,1,3. 
       -- For DENSE_RANK(), you get 1,1,2
       DENSE_RANK() OVER(ORDER BY hire_date) AS rank
FROM employees
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,employee_id,first_name,hire_date,department,rank
0,271,Norbie,2003-01-01,First Aid,1
1,300,Cassandra,2003-01-01,Beauty,1
2,79,Rora,2003-01-12,Children Clothing,2
3,5,Feliks,2003-01-14,Computers,3
4,739,Cecilius,2003-01-20,Vitamins,4
5,488,Eugenius,2003-01-26,Toys,5
6,75,Fiorenze,2003-02-17,Phones & Tablets,6
7,83,Elnora,2003-02-22,First Aid,7
8,744,Chelsey,2003-02-24,First Aid,8
9,814,Dorothea,2003-02-27,Grocery,9


In [ ]:
# Retrieve the employee_id, first_name, hire_date of employees for different departments
query = """
SELECT employee_id,
       first_name,
       hire_date,
       department,
       -- Now, partition by department, to get the info of employees per department numbered and ordered by hire date
       ROW_NUMBER() OVER(
            PARTITION BY department
            ORDER BY hire_date) AS Row_N
FROM employees
LIMIT 40;
"""

pd.read_sql_query(query, conn)

,employee_id,first_name,hire_date,department,Row_N
0,927,Maryellen,2003-04-19,Automotive,1
1,840,Archibold,2003-04-26,Automotive,2
2,988,Tabb,2003-05-02,Automotive,3
3,648,Abbott,2003-06-05,Automotive,4
4,305,Ladonna,2003-08-10,Automotive,5
5,126,Roslyn,2003-08-11,Automotive,6
6,274,Lorelle,2004-01-27,Automotive,7
7,515,Cybil,2004-04-01,Automotive,8
8,249,Sterling,2004-09-02,Automotive,9
9,456,Jessalyn,2004-10-03,Automotive,10


In [24]:
query = """ 
-- Retrieve the hire_date. Return details of employees hired on or before 31st Dec, 2005 and are in First Aid, Movies and Computers departments

SELECT first_name, 
       email, 
       department, 
       salary,
       hire_date,
       RANK() OVER (
            PARTITION BY department
            ORDER BY salary DESC) AS ranked_salary
FROM employees
WHERE hire_date <= '2005-12-31' AND
      department IN ('First Aid', 'Movies', 'Computers');
"""

pd.read_sql_query(query, conn)

,first_name,email,department,salary,hire_date,ranked_salary
0,Parnell,pdavidoff8i@zimbio.com,Computers,155000,2004-03-29,1
1,Webb,NaN,Computers,152504,2005-10-13,2
2,Javier,jrosendale98@ca.gov,Computers,87199,2003-11-12,3
3,Retha,rdebneyak@indiegogo.com,Computers,74771,2004-06-29,4
4,Feliks,fmorffew4@a8.net,Computers,55307,2003-01-14,5
5,Kimberley,NaN,Computers,49788,2004-01-08,6
6,Levi,NaN,Computers,35076,2003-10-21,7
7,Iver,NaN,Computers,21626,2004-12-30,8
8,Orsa,otournel5@webmd.com,First Aid,163239,2005-07-19,1
9,Stacee,spawellek7e@fotki.com,First Aid,136707,2004-04-21,2


In [25]:
query = """
-- This returns how many employees are in each department
SELECT department, 
       COUNT(*) dept_count
FROM employees
GROUP BY department
ORDER BY dept_count DESC;
"""

pd.read_sql_query(query, conn)

,department,dept_count
0,First Aid,58
1,Movies,56
2,Device Repair,51
3,Clothing,49
4,Toys,47
5,Computers,47
6,Children Clothing,47
7,Beauty,45
8,Furniture,43
9,Jewelry,41


In [26]:
## Return the fifth ranked salary for each department
query = """
WITH ranked_salary_cte AS (
    SELECT first_name, 
            email, 
            department, 
            salary,
            hire_date,
            RANK() OVER (
                PARTITION BY department
                ORDER BY salary DESC) AS ranked_salary
FROM employees
WHERE hire_date <= '2005-12-31' AND
      department IN ('First Aid', 'Movies', 'Computers'))

SELECT * FROM ranked_salary_cte
WHERE ranked_salary = 5;
"""

pd.read_sql_query(query, conn)

,first_name,email,department,salary,hire_date,ranked_salary
0,Feliks,fmorffew4@a8.net,Computers,55307,2003-01-14,5
1,Leland,lreichartzpz@webeden.co.uk,First Aid,131935,2005-06-04,5
2,Granny,NaN,Movies,125798,2003-05-13,5


In [29]:
# 2nd way using subqueries
query = """
SELECT *
FROM (SELECT first_name, 
             email, 
             department,
             salary,
             DENSE_RANK() OVER (
                     PARTITION BY department
                     ORDER BY salary DESC) AS ranked_salary
       FROM employees
       ) AS sub
WHERE ranked_salary = 5;
"""
pd.read_sql_query(query, conn)

,first_name,email,department,salary,ranked_salary
0,Betsey,breedshawc5@phoca.cz,Automotive,152141,5
1,Garald,NaN,Beauty,145225,5
2,Christine,chessing2y@dailymail.co.uk,Books,149864,5
3,Kevin,kschubart9v@dailymotion.com,Camping,154856,5
4,Nicolis,nvigersdo@t.co,Children Clothing,137567,5
5,Hester,hseakin27@netlog.com,Clothing,150887,5
6,Laryssa,lmumns@shinystat.com,Computers,152831,5
7,Gabriel,gwidocks5i@acquirethisname.com,Cosmetics,154376,5
8,Gorden,NaN,Decor,146577,5
9,Sebastian,slefeuvre6d@abc.net.au,Device Repair,157861,5


In [34]:
# Create a CTE
query = """
WITH purchase_count AS (
SELECT Customer_ID, COUNT(Sales) as purchase_N 
FROM sales
GROUP BY Customer_ID)

-- Difference between ROW_NUMBER(), RANK(), and DENSE_RANK()
SELECT *,
       ROW_NUMBER() OVER(ORDER BY purchase_N DESC) AS Row_N,
       RANK() OVER(ORDER BY purchase_N DESC) AS rank_N,
       DENSE_RANK() OVER(ORDER BY purchase_N DESC) AS dense_rank_N
FROM purchase_count
LIMIT 40;
"""
pd.read_sql_query(query, conn)

,Customer_ID,purchase_N,Row_N,rank_N,dense_rank_N
0,WB-21850,37,1,1,1
1,JL-15835,34,2,2,2
2,MA-17560,34,3,2,2
3,PP-18955,34,4,2,2
4,CK-12205,32,5,5,3
5,EH-13765,32,6,5,3
6,JD-15895,32,7,5,3
7,SV-20365,32,8,5,3
8,AP-10915,31,9,9,4
9,EP-13915,31,10,9,4


**NTILE Window Function**

NTILE() is used to break or page the result set into groups!  

In [ ]:
# Group the employees table into five groups based on the order of their salaries
query = """
SELECT first_name, 
       department, 
       salary,
       NTILE(5) OVER(ORDER BY salary DESC) AS grouped_salaries
FROM employees;
"""
pd.read_sql_query(query, conn)  # Each group has 200 records

,first_name,department,salary,grouped_salaries
0,Jacklyn,Clothing,166976,1
1,Carissa,Music,166765,1
2,Riley,Camping,166569,1
3,Lauren,Pharmacy,166016,1
4,Lucy,Sports,165660,1
...,...,...,...,...
995,Roarke,Device Repair,21023,5
996,Marylin,Music,20776,5
997,Roger,Device Repair,20664,5
998,Addia,Grocery,20613,5


In [ ]:
# Now, group the employees table into five groups for each department based on the order of their salaries
query = """
SELECT first_name, 
       department, 
       salary,
       NTILE(5) OVER(
            PARTITION BY department
            ORDER BY salary DESC) AS grouped_salaries
FROM employees
LIMIT 40;
"""
pd.read_sql_query(query, conn)  # Each department is split into 5 groups

,first_name,department,salary,grouped_salaries
0,Mill,Automotive,162522,1
1,Irita,Automotive,160783,1
2,Tammie,Automotive,160039,1
3,Roslyn,Automotive,157260,1
4,Betsey,Automotive,152141,1
5,Cherianne,Automotive,150821,1
6,Chrissy,Automotive,146522,1
7,Poppy,Automotive,144511,2
8,Cy,Automotive,144146,2
9,Jilleen,Automotive,137393,2


In [39]:
# Create a CTE that returns details of an employee and group the employees into five groups based on the order of their salaries
# Then find the average salary for each group of employees
query = """
WITH salary_ranks AS (
        SELECT first_name, 
               email,
               department, 
               salary,
               NTILE(5) OVER (ORDER BY salary DESC) AS rank_of_salary
        FROM employees
)
SELECT rank_of_salary,
       ROUND(AVG(salary)) AS avg_salary
FROM salary_ranks
GROUP BY rank_of_salary;
"""
pd.read_sql_query(query, conn)

,rank_of_salary,avg_salary
0,1,151648.0
1,2,119197.0
2,3,90136.0
3,4,62086.0
4,5,34791.0


**Aggregate Window Functions**

In [40]:
query = """
-- This returns how many employees are in each department
SELECT department, 
       COUNT(*) AS dept_count
FROM employees
GROUP BY department
ORDER BY department;
"""
pd.read_sql_query(query, conn)

,department,dept_count
0,Automotive,32
1,Beauty,45
2,Books,37
3,Camping,36
4,Children Clothing,47
5,Clothing,49
6,Computers,47
7,Cosmetics,34
8,Decor,39
9,Device Repair,51


In [46]:
# Retrieve the first names, department and number of employees working in that department
query = """
SELECT first_name,
       department,
       COUNT(*) OVER(PARTITION BY department) dept_count
FROM employees;
"""
pd.read_sql_query(query, conn)

,first_name,department,dept_count
0,Vanda,Automotive,32
1,Irita,Automotive,32
2,Jilleen,Automotive,32
3,Roslyn,Automotive,32
4,Tammie,Automotive,32
...,...,...,...
995,Lion,Vitamins,37
996,Evin,Vitamins,37
997,Odessa,Vitamins,37
998,Tab,Vitamins,37


In [61]:
# Total Salary for all employees
query = """
SELECT first_name,
       department, 
       hire_date,
       salary,
       SUM(salary) OVER (PARTITION BY department) AS dept_total_salary,
       COUNT(*) OVER (PARTITION BY department) dept_count
FROM employees
ORDER BY dept_total_salary DESC;
"""
pd.read_sql_query(query, conn)

,first_name,department,hire_date,salary,dept_total_salary,dept_count
0,Dayle,First Aid,2003-03-01,82753,5170963,58
1,Aldon,First Aid,2013-06-30,109592,5170963,58
2,Reuben,First Aid,2009-03-14,34572,5170963,58
3,Elnora,First Aid,2003-02-22,34355,5170963,58
4,Elwin,First Aid,2014-07-09,141261,5170963,58
...,...,...,...,...,...,...
995,Kincaid,Security,2010-11-10,162233,539701,6
996,Rorke,Security,2005-01-19,89252,539701,6
997,Gianni,Security,2008-11-20,38963,539701,6
998,Tallie,Security,2006-08-07,146932,539701,6


In [63]:
query = """
-- Total Salary for each department and order by the hire date. Call the new column running_total
SELECT first_name, hire_date, department, salary,
SUM(salary) OVER(PARTITION BY department
				 ORDER BY hire_date) AS running_total
FROM employees
LIMIT 40;
"""
pd.read_sql_query(query, conn)

,first_name,hire_date,department,salary,running_total
0,Maryellen,2003-04-19,Automotive,115973,115973
1,Archibold,2003-04-26,Automotive,69379,185352
2,Tabb,2003-05-02,Automotive,47591,232943
3,Abbott,2003-06-05,Automotive,106517,339460
4,Ladonna,2003-08-10,Automotive,111775,451235
5,Roslyn,2003-08-11,Automotive,157260,608495
6,Lorelle,2004-01-27,Automotive,119959,728454
7,Cybil,2004-04-01,Automotive,123828,852282
8,Sterling,2004-09-02,Automotive,56095,908377
9,Jessalyn,2004-10-03,Automotive,86929,995306


In [64]:
query = """
SELECT first_name,
       department, 
       hire_date,
       salary,
       SUM(salary) OVER (ORDER BY hire_date) AS total_salary
FROM employees
LIMIT 30;
"""
pd.read_sql_query(query, conn) 

,first_name,department,hire_date,salary,total_salary
0,Norbie,First Aid,2003-01-01,82215,189151
1,Cassandra,Beauty,2003-01-01,106936,189151
2,Rora,Children Clothing,2003-01-12,153489,342640
3,Feliks,Computers,2003-01-14,55307,397947
4,Cecilius,Vitamins,2003-01-20,98882,496829
5,Eugenius,Toys,2003-01-26,152118,648947
6,Fiorenze,Phones & Tablets,2003-02-17,51266,700213
7,Elnora,First Aid,2003-02-22,34355,734568
8,Chelsey,First Aid,2003-02-24,57309,791877
9,Dorothea,Grocery,2003-02-27,46062,837939


In [67]:
query = """
-- Retrieve the different region ids
SELECT DISTINCT region_id
FROM employees
ORDER BY region_id;
"""
pd.read_sql_query(query, conn)

,region_id
0,1
1,2
2,3
3,4
4,5
5,6
6,7


In [81]:
query = """
SELECT first_name,
       department,
       COUNT(*) OVER (PARTITION BY department) AS dept_count,
       region_id,
       COUNT(*) OVER (PARTITION BY region_id) AS region_count
FROM employees
LIMIT 40;
"""
pd.read_sql_query(query, conn) 

,first_name,department,dept_count,region_id,region_count
0,Irita,Automotive,32,1,152
1,Roslyn,Automotive,32,1,152
2,Doe,Automotive,32,1,152
3,Clementina,Automotive,32,1,152
4,Cybil,Automotive,32,1,152
5,Chrissy,Automotive,32,1,152
6,Merlina,Automotive,32,1,152
7,Ladonna,Automotive,32,2,141
8,Betsey,Automotive,32,2,141
9,Poppy,Automotive,32,2,141


In [82]:
query = """
-- Retrieve the first names, department and number of employees working in that department and in region 2
SELECT first_name, department, 
COUNT(*) OVER(PARTITION BY department) AS dept_count
FROM employees
WHERE region_id = 2;
"""
pd.read_sql_query(query, conn)

,first_name,department,dept_count
0,Ladonna,Automotive,4
1,Betsey,Automotive,4
2,Poppy,Automotive,4
3,Archibold,Automotive,4
4,Willabella,Beauty,12
...,...,...,...
136,Archambault,Vitamins,5
137,Ame,Vitamins,5
138,Elvera,Vitamins,5
139,Shell,Vitamins,5


In [83]:
query = """
-- Create a common table expression to retrieve the customer_id, ship_mode, and how many times the customer has purchased from the mall
WITH purchase_count AS (
        SELECT customer_id, 
               ship_mode, 
               COUNT(sales) AS purchase
        FROM sales
        GROUP BY customer_id, ship_mode
        ORDER BY purchase DESC
)

-- Calculate the cumulative sum of customers purchase for the different ship mode
SELECT customer_id, 
       ship_mode, 
       purchase, 
SUM(purchase) OVER(PARTITION BY ship_mode
				   ORDER BY customer_id) AS sum_of_sales
FROM purchase_count;
"""
pd.read_sql_query(query, conn)

,customer_id,ship_mode,purchase,sum_of_sales
0,AA-10315,First Class,1,1
1,AA-10375,First Class,4,5
2,AA-10645,First Class,7,12
3,AB-10015,First Class,5,17
4,AB-10060,First Class,8,25
...,...,...,...,...
2040,XP-21865,Standard Class,23,5923
2041,YC-21895,Standard Class,5,5928
2042,YS-21880,Standard Class,11,5939
2043,ZC-21910,Standard Class,26,5965


**Window Frames**

In [84]:
query = """
SELECT first_name,
       hire_date,
       salary,
       SUM(salary) OVER(ORDER BY hire_date
                        RANGE BETWEEN UNBOUNDED PRECEDING
                        AND CURRENT ROW) AS running_total
FROM employees;
"""
pd.read_sql_query(query, conn)

,first_name,hire_date,salary,running_total
0,Norbie,2003-01-01,82215,189151
1,Cassandra,2003-01-01,106936,189151
2,Rora,2003-01-12,153489,342640
3,Feliks,2003-01-14,55307,397947
4,Cecilius,2003-01-20,98882,496829
...,...,...,...,...
995,Eloisa,2016-12-02,39200,91125583
996,Edik,2016-12-11,88378,91213961
997,Roxie,2016-12-16,42224,91256185
998,Cherianne,2016-12-18,150821,91407006


In [85]:
query = """
SELECT first_name,
       hire_date,
       salary,
       SUM(salary) OVER(ORDER BY hire_date
                        ROWS BETWEEN 1 PRECEDING
                        AND CURRENT ROW) AS running_total
FROM employees;
"""
pd.read_sql_query(query, conn)

,first_name,hire_date,salary,running_total
0,Norbie,2003-01-01,82215,82215
1,Cassandra,2003-01-01,106936,189151
2,Rora,2003-01-12,153489,260425
3,Feliks,2003-01-14,55307,208796
4,Cecilius,2003-01-20,98882,154189
...,...,...,...,...
995,Eloisa,2016-12-02,39200,105584
996,Edik,2016-12-11,88378,127578
997,Roxie,2016-12-16,42224,130602
998,Cherianne,2016-12-18,150821,193045


In [87]:
# Moving average
query = """
SELECT first_name,
       hire_date,
       salary,
       ROUND(AVG(salary) OVER(ORDER BY hire_date
                        ROWS BETWEEN 2 PRECEDING
                        AND CURRENT ROW)) AS running_average
FROM employees;
"""
pd.read_sql_query(query, conn)

,first_name,hire_date,salary,running_average
0,Norbie,2003-01-01,82215,82215.0
1,Cassandra,2003-01-01,106936,94576.0
2,Rora,2003-01-12,153489,114213.0
3,Feliks,2003-01-14,55307,105244.0
4,Cecilius,2003-01-20,98882,102559.0
...,...,...,...,...
995,Eloisa,2016-12-02,39200,44387.0
996,Edik,2016-12-11,88378,64654.0
997,Roxie,2016-12-16,42224,56601.0
998,Cherianne,2016-12-18,150821,93808.0


In [89]:
query = """
SELECT first_name, hire_date, salary,
SUM(salary) OVER(ORDER BY hire_date 
				 ROWS BETWEEN
				 3 PRECEDING AND CURRENT ROW) AS running_total
FROM employees;
"""
pd.read_sql_query(query, conn)

,first_name,hire_date,salary,running_total
0,Norbie,2003-01-01,82215,82215
1,Cassandra,2003-01-01,106936,189151
2,Rora,2003-01-12,153489,342640
3,Feliks,2003-01-14,55307,397947
4,Cecilius,2003-01-20,98882,414614
...,...,...,...,...
995,Eloisa,2016-12-02,39200,271655
996,Edik,2016-12-11,88378,221540
997,Roxie,2016-12-16,42224,236186
998,Cherianne,2016-12-18,150821,320623


In [ ]:
# Retrieve the last value of the department
query = """
SELECT department, division,
FIRST_VALUE(department) OVER(ORDER BY department ASC) first_department,
LAST_VALUE(department) OVER(ORDER BY department
                            RANGE BETWEEN 
                            UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_department
FROM departments;
"""
pd.read_sql_query(query, conn)

,department,division,first_department,last_department
0,Automotive,Hardware,Automotive,Vitamins
1,Beauty,Fashion,Automotive,Vitamins
2,Books,Entertainment,Automotive,Vitamins
3,Camping & Fishing,Outdoors,Automotive,Vitamins
4,Children Clothing,Kids,Automotive,Vitamins
5,Clothing,Home,Automotive,Vitamins
6,Computers,Electronics,Automotive,Vitamins
7,Cosmetics,Fashion,Automotive,Vitamins
8,Decor,Home,Automotive,Vitamins
9,Device Repair,Electronics,Automotive,Vitamins


In [95]:
query = """
-- Create a common table expression to retrieve the customer_id, ship_mode, and how many times the customer has purchased from the mall
WITH purchase_count AS (
SELECT customer_id, 
       COUNT(sales) AS purchase
FROM sales
GROUP BY customer_id
ORDER BY purchase DESC
)

SELECT customer_id, purchase, 
MAX(purchase) OVER(ORDER BY customer_id ASC) AS max_of_sales,
MAX(purchase) OVER(ORDER BY customer_id ASC
				  ROWS BETWEEN
				  CURRENT ROW AND 1 FOLLOWING) AS next_max_of_sales
FROM purchase_count
LIMIT 40;
"""
pd.read_sql_query(query, conn)

,customer_id,purchase,max_of_sales,next_max_of_sales
0,AA-10315,11,11,15
1,AA-10375,15,15,15
2,AA-10480,12,15,18
3,AA-10645,18,18,18
4,AB-10015,6,18,18
5,AB-10060,18,18,20
6,AB-10105,20,20,20
7,AB-10150,12,20,14
8,AB-10165,14,20,14
9,AB-10255,14,20,14


**GROUPING SETS(), ROLLUP(), and CUBE()**

In [97]:
query = """
-- Find the sum of the quantity for different ship modes
SELECT ship_mode, SUM(quantity) AS quantity_per_mode
FROM sales
GROUP BY ship_mode;
"""
pd.read_sql_query(query, conn)

,Ship_Mode,quantity_per_mode
0,First Class,5693
1,Same Day,1960
2,Second Class,7423
3,Standard Class,22797


In [98]:
query = """
-- Find the sum of the quantity for different ship modes
SELECT category, SUM(quantity) AS quantity_per_mode
FROM sales
GROUP BY category;
"""
pd.read_sql_query(query, conn)

,Category,quantity_per_mode
0,Furniture,8028
1,Office Supplies,22906
2,Technology,6939


In [99]:
query = """
-- Find the sum of the quantity for different ship modes
SELECT sub_category, SUM(quantity) AS quantity_per_mode
FROM sales
GROUP BY sub_category;
"""
pd.read_sql_query(query, conn)

,Sub_Category,quantity_per_mode
0,Accessories,2976
1,Appliances,1729
2,Art,3000
3,Binders,5974
4,Bookcases,868
5,Chairs,2356
6,Copiers,234
7,Envelopes,906
8,Fasteners,914
9,Furnishings,3563


In [103]:
# Alternative to using GROUPING SETS in SQLite
query = """
SELECT ship_mode, NULL AS category, NULL AS sub_category, SUM(quantity) AS total_quantity
FROM sales
GROUP BY ship_mode

UNION ALL

SELECT NULL, category, NULL, SUM(quantity)
FROM sales
GROUP BY category

UNION ALL

SELECT NULL, NULL, sub_category, SUM(quantity)
FROM sales
GROUP BY sub_category;
"""
pd.read_sql_query(query, conn)

,Ship_Mode,category,sub_category,total_quantity
0,First Class,NaN,NaN,5693
1,Same Day,NaN,NaN,1960
2,Second Class,NaN,NaN,7423
3,Standard Class,NaN,NaN,22797
4,NaN,Furniture,NaN,8028
5,NaN,Office Supplies,NaN,22906
6,NaN,Technology,NaN,6939
7,NaN,NaN,Accessories,2976
8,NaN,NaN,Appliances,1729
9,NaN,NaN,Art,3000


In [114]:
# ROLLUP() Alternative in SQLITE
query = """
-- Level 3: grouped by all three columns
SELECT category, sub_category, ship_mode, SUM(quantity) AS total_quantity
FROM sales
GROUP BY category, sub_category, ship_mode

UNION ALL

-- Level 2: grouped by the first two only
SELECT category, sub_category, NULL AS ship_mode, SUM(quantity)
FROM sales
GROUP BY category, sub_category

UNION ALL

-- Level 1: grouped by the first column only
SELECT category, NULL AS sub_category, NULL AS ship_mode, SUM(quantity)
FROM sales
GROUP BY category

UNION ALL

-- Level 0: grand total, no grouping at all
SELECT NULL, NULL, NULL, SUM(quantity)
FROM sales
;
"""
pd.read_sql_query(query, conn)

,Category,Sub_Category,Ship_Mode,total_quantity
0,Furniture,Bookcases,First Class,187
1,Furniture,Bookcases,Same Day,19
2,Furniture,Bookcases,Second Class,175
3,Furniture,Bookcases,Standard Class,487
4,Furniture,Chairs,First Class,335
...,...,...,...,...
84,Technology,Phones,NaN,3289
85,Furniture,NaN,NaN,8028
86,Office Supplies,NaN,NaN,22906
87,Technology,NaN,NaN,6939


In [ ]:
# Alternative to CUBE()-  produces subtotals and grand totals for every permutation of the columns provided.
query = """
-- (category, sub_category, ship_mode)
SELECT category, sub_category, ship_mode, SUM(quantity) AS total_quantity
FROM sales GROUP BY category, sub_category, ship_mode

UNION ALL

-- (category, sub_category)
SELECT category, sub_category, NULL, SUM(quantity)
FROM sales GROUP BY category, sub_category

UNION ALL

-- (category, ship_mode)
SELECT category, NULL, ship_mode, SUM(quantity)
FROM sales GROUP BY category, ship_mode

UNION ALL

-- (sub_category, ship_mode)
SELECT NULL, sub_category, ship_mode, SUM(quantity)
FROM sales GROUP BY sub_category, ship_mode

UNION ALL

-- (category) alone
SELECT category, NULL, NULL, SUM(quantity)
FROM sales GROUP BY category

UNION ALL

-- (sub_category) alone
SELECT NULL, sub_category, NULL, SUM(quantity)
FROM sales GROUP BY sub_category

UNION ALL

-- (ship_mode) alone
SELECT NULL, NULL, ship_mode, SUM(quantity)
FROM sales GROUP BY ship_mode

UNION ALL

-- () grand total
SELECT NULL, NULL, NULL, SUM(quantity)
FROM sales
LIMIT 100;
"""
pd.read_sql_query(query, conn)

,Category,Sub_Category,Ship_Mode,total_quantity
0,Furniture,Bookcases,First Class,187
1,Furniture,Bookcases,Same Day,19
2,Furniture,Bookcases,Second Class,175
3,Furniture,Bookcases,Standard Class,487
4,Furniture,Chairs,First Class,335
...,...,...,...,...
95,Technology,NaN,Second Class,1374
96,Technology,NaN,Standard Class,4122
97,NaN,Accessories,First Class,461
98,NaN,Accessories,Same Day,166
